Import Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

Synthetic Dataset (Sequence Reversal)

In [3]:
def generate_data(n_samples=10000, seq_len=5):

    X = np.random.randint(1,10,(n_samples,seq_len))
    y = np.flip(X, axis=1).copy()

    return torch.LongTensor(X), torch.LongTensor(y)

X, y = generate_data()
print(X.shape, y.shape)

torch.Size([10000, 5]) torch.Size([10000, 5])


Embedding Layer

In [4]:
class TokenEmbedding(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))

    def forward(self, x):
        return self.embedding[x]

Position embedding

In [5]:
class PositionEmbedding(nn.Module):

    def __init__(self, max_len, d_model):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(max_len, d_model))

    def forward(self, x):

        seq_len = x.shape[1]
        return self.embedding[:seq_len]

Self Attention

In [6]:
class SelfAttention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

    def forward(self, x):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / np.sqrt(Q.size(-1))

        weights = torch.softmax(scores, dim=-1)

        return torch.matmul(weights, V)

Feed Forward Network

In [7]:
class FeedForward(nn.Module):

    def __init__(self, d_model, hidden):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d_model)
        )

    def forward(self, x):
        return self.net(x)

Transformer Encoder Block

In [8]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.attn = SelfAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = FeedForward(d_model, d_model*2)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))

        return x

Full Transformer

In [9]:
class Transformer(nn.Module):

    def __init__(self, vocab_size=10, seq_len=5, d_model=32):

        super().__init__()

        self.token_embed = TokenEmbedding(vocab_size, d_model)
        self.pos_embed = PositionEmbedding(seq_len, d_model)

        self.encoder = EncoderBlock(d_model)

        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        tok = self.token_embed(x)
        pos = self.pos_embed(x)

        x = tok + pos

        x = self.encoder(x)

        return self.output(x)

Training Loop

In [10]:
model = Transformer()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    optimizer.zero_grad()

    out = model(X)

    loss = criterion(out.view(-1,10), y.view(-1))

    loss.backward()

    optimizer.step()

    print("Epoch", epoch, "Loss:", loss.item())

Epoch 0 Loss: 2.421725273132324
Epoch 1 Loss: 2.39786696434021
Epoch 2 Loss: 2.3754186630249023
Epoch 3 Loss: 2.35432505607605
Epoch 4 Loss: 2.3345024585723877
Epoch 5 Loss: 2.3158469200134277
Epoch 6 Loss: 2.298226833343506
Epoch 7 Loss: 2.28151535987854
Epoch 8 Loss: 2.2655913829803467
Epoch 9 Loss: 2.2503364086151123
Epoch 10 Loss: 2.2356467247009277
Epoch 11 Loss: 2.221423625946045
Epoch 12 Loss: 2.207576274871826
Epoch 13 Loss: 2.194023370742798
Epoch 14 Loss: 2.180691957473755
Epoch 15 Loss: 2.16752290725708
Epoch 16 Loss: 2.154465436935425
Epoch 17 Loss: 2.1414661407470703
Epoch 18 Loss: 2.1284778118133545
Epoch 19 Loss: 2.1154513359069824


Generate Output

In [11]:
def predict(model, seq):

    seq = torch.LongTensor(seq).unsqueeze(0)

    with torch.no_grad():
        out = model(seq)

    return out.argmax(-1).squeeze().numpy()

Test

In [12]:
test = [3,5,2,4,1]

print("Input:", test)
print("Prediction:", predict(model,test))

Input: [3, 5, 2, 4, 1]
Prediction: [5 5 2 4 1]
